# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank.ai_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Binary Classification → used to produce a Ranked Score**

The core task is **binary classification**: predict whether a page is declining (`is_declining_label = True/False`). However, the deliverable is not a binary flag — it is a **ranked refresh queue** ordered by urgency. So the model outputs a probability score (`P(declining)`), and pages are ranked highest-first by that probability. This gives content teams a prioritized action list, not just a yes/no.

Why classification and not pure ranking (LTR)? Because we have a natural binary label from the data pipeline (`trend_direction == 'down'`). We don't need to compare pairs of documents — we need each page to get an independent risk score. Binary classification with probability output achieves both: it trains on the label, and its output probability is used as a ranking score.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label`** = `(trend_direction == 'down')`

This is a **rule-based proxy label**, not a directly observed outcome. The true outcome we care about — "will this page lose significant search traffic over the next quarter if we don't refresh it?" — is not in the dataset. Instead, we use a retrospective rule applied to the 90-day trailing window: `trend_direction` is computed by comparing the last 30 days vs the preceding 30 days of impressions/clicks/sessions.

**Limitations of this proxy:**
- It looks backward (past 90 days), not forward. A page that just started declining may not yet cross the threshold.
- Short-term seasonal dips fire the label even for healthy pages.
- The rule is symmetric: a small 2% drop and a severe 80% drop both produce label=True.

**What we must never use as features:** `trend_direction` and `trend_pct` — these are the direct ingredients of the label and would constitute target leakage.

## 3. Train / test split design

*How do you split? Why is that design honest for this question?*

**Split: Client-grouped holdout (20% of clients withheld)**

The split holds out ~6–7 clients entirely from training (the dataset has 32 clients). All pages from held-out clients appear only in the test set. This is the approach used by the reference pipeline (`03_train_model.py`).

**Why this design is honest:**
- Pages from the same client are highly correlated — same content strategy, same industry, same GA4 setup. A random row split would let the model memorize client-level patterns and appear to generalize when it has actually seen the client before.
- The business question is: "Does this model work for a new client FlyRank onboards next month?" Client-holdout honestly simulates that.
- Alternative (time-based split) is not possible here because all rows are from the same 90-day window — there is no time ordering of rows to exploit.

**Implementation sketch:**
```python
from sklearn.model_selection import GroupShuffleSplit
X = ...  # feature matrix
y = ...  # labels
groups = df['client_id']
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
```

## 4. Success metric and why

*One primary metric. Why does it fit this problem better than accuracy?*

**Primary metric: Precision@50 (from the reference pipeline) + ROC-AUC as secondary**

**Precision@50:** Of the top-50 pages the model flags as most-urgently-declining, what fraction are truly declining? This directly mirrors the business workflow: a content team reviews 50 pages per sprint. We want the list to be as accurate as possible.

**Why not accuracy?** The dataset is moderately imbalanced (~40–60% declining). Accuracy can be high by predicting the majority class. What matters is whether the *top of the ranked list* is trustworthy.

**Why not pure recall?** Flagging all pages as declining gives 100% recall — useless for prioritization.

**ROC-AUC** is used as a secondary metric because it measures ranking quality across all thresholds, independent of the operating point. It is also threshold-invariant and robust to class imbalance.

**Baseline to beat:** The reference hand-rule baseline's Precision@50. The model must improve on this to justify the added complexity.

In [1]:
# Load the data and verify the label
import pandas as pd
import numpy as np
import os

if os.path.exists('data/raw/content_refresh_anonymized.csv'):
    df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
elif os.path.exists('../data/raw/content_refresh_anonymized.csv'):
    df = pd.read_csv('../data/raw/content_refresh_anonymized.csv')

# Construct the label
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print('Label value counts:')
print(df['is_declining_label'].value_counts())
print(f'\nClass balance: {df["is_declining_label"].mean():.1%} declining')
print(f'\nClient distribution across label:')
print(df.groupby('is_declining_label')['client_id'].nunique())

# Verify split design — how many rows per client?
print('\nRows per client (sample):')
print(df.groupby('client_id').size().describe())

Label value counts:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Class balance: 54.2% declining

Client distribution across label:
is_declining_label
0    32
1    30
Name: client_id, dtype: int64

Rows per client (sample):
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64
